# SVM Baseline for AI-Generated Text Detection #

## 0. Goal ##
This notebook implements TF-IDF + LinearSVC for: 
- binary human-vs-AI detection
- AI model attribution
- cross-dataset generalization

## 1. Smoke-test the data loader ##

In [ ]:
# Setup
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results" / "svm"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

In [4]:
# Import loader
import pandas as pd
import numpy as np

from data_provider.data_loader import (
    load_dataset,
    make_binary_dataset,
    make_multiclass_dataset,
)

In [5]:
df = load_dataset(foldername=DATA_DIR)
df.head()

,text,binary_label,source_model,dataset_name,domain,original_label
0,Cars. Cars have been around since they became ...,human,human,ai_vs_human_text,unknown,0.0
1,Transportation is a large necessity in most co...,human,human,ai_vs_human_text,unknown,0.0
2,"""America's love affair with it's vehicles seem...",human,human,ai_vs_human_text,unknown,0.0
3,How often do you ride in a car? Do you drive a...,human,human,ai_vs_human_text,unknown,0.0
4,Cars are a wonderful thing. They are perhaps o...,human,human,ai_vs_human_text,unknown,0.0


In [6]:
# Sanity checks
print(df.shape)
print(df.columns)

display(df.head())
display(df["dataset_name"].value_counts())
display(df["binary_label"].value_counts())
display(df["source_model"].value_counts())

(478799, 6)
Index(['text', 'binary_label', 'source_model', 'dataset_name', 'domain',
       'original_label'],
      dtype='object')


,text,binary_label,source_model,dataset_name,domain,original_label
0,Cars. Cars have been around since they became ...,human,human,ai_vs_human_text,unknown,0.0
1,Transportation is a large necessity in most co...,human,human,ai_vs_human_text,unknown,0.0
2,"""America's love affair with it's vehicles seem...",human,human,ai_vs_human_text,unknown,0.0
3,How often do you ride in a car? Do you drive a...,human,human,ai_vs_human_text,unknown,0.0
4,Cars are a wonderful thing. They are perhaps o...,human,human,ai_vs_human_text,unknown,0.0


dataset_name
ai_vs_human_text             465199
ai_text_detection_dataset     10129
daigt_v2_train                 2971
multi_model_detection           500
Name: count, dtype: Int64

binary_label
human    285597
ai       193202
Name: count, dtype: Int64

source_model
human                                 285597
unknown_ai                            189912
llama2_chat                             1754
darragh_claude_v6                        406
NousResearch/Llama-2-7b-chat-hf          297
palm-text-bison1                         196
cohere-command                           161
mistralai/Mistral-7B-Instruct-v0.1       156
Gemini-2_Pro                             118
GPT-5_Preview                            107
Claude-4_Opus                             95
Name: count, dtype: Int64

## 2. Add notebook-level audit checks ##

In [7]:
# Missing values
df.isna().sum()

# Check for duplicate text after loading
df["text"].duplicated().sum()

# Check text lengths
df["num_chars"] = df["text"].str.len()
df["num_words"] = df["text"].str.split().str.len()

display(df.groupby("binary_label")[["num_chars", "num_words"]].describe())
display(df.groupby("dataset_name")[["num_chars", "num_words"]].describe())

num_chars                                                  \
                 count         mean          std   min     25%     50%   
binary_label                                                             
ai            193202.0  2121.331798   785.874158   1.0  1653.0  2049.0   
human         285597.0  2356.851448  1088.997213  76.0  1538.0  2157.0   

                             num_words                                       \
                 75%     max     count        mean         std   min    25%   
binary_label                                                                  
ai            2482.0  8857.0  193202.0  344.302621  117.160317   1.0  274.0   
human         2928.0  9180.0  285597.0  423.000074  188.420512  13.0  281.0   

                                    
                50%    75%     max  
binary_label                        
ai            338.0  404.0  1238.0  
human         390.0  522.0  1668.0

num_chars                                          \
                              count         mean         std    min     25%   
dataset_name                                                                  
ai_text_detection_dataset   10129.0  2014.998420  810.020969    7.0  1373.0   
ai_vs_human_text           465199.0  2271.181742  988.220716    1.0  1587.0   
daigt_v2_train               2971.0  1970.788960  439.945313  508.0  1719.5   
multi_model_detection         500.0   277.282000   84.772964   96.0   220.0   

                                                   num_words              \
                              50%      75%     max     count        mean   
dataset_name                                                               
ai_text_detection_dataset  2099.0  2573.00  5990.0   10129.0  331.749729   
ai_vs_human_text           2103.0  2722.00  9180.0  465199.0  393.266780   
daigt_v2_train             1879.0  2101.00  4445.0    2971.0  337.524403   
multi_model_detection       268.0   326.25   608.0     500.0   34.232000   

                                                                            
                                  std   min     25%    50%     75%     max  
dataset_name                                                                
ai_text_detection_dataset  127.049050   2.0  236.00  344.0  415.00   998.0  
ai_vs_human_text           168.559918   1.0  278.00  362.0  471.00  1668.0  
daigt_v2_train              68.720100   4.0  294.00  326.0  364.00   713.0  
multi_model_detection        9.448212  13.0   27.75   34.0   40.25    64.0

In [8]:
pd.set_option("display.max_colwidth", 300)

display(df.sample(10, random_state=RANDOM_STATE))

,text,binary_label,source_model,dataset_name,domain,original_label,num_chars,num_words
477305,"The Face on Mars is a natural landform. This is evidenced by the fact that the Face has been eroded over time by the Martian elements. The Face is not a perfect symmetrical structure, as would be expected if it were created by aliens. The Face is also not aligned with the Martian grid, which is ...",ai,unknown_ai,ai_text_detection_dataset,unknown,1,345,64
434853,"The electoral college process consist of the selection of the elector, the meeting of the electors where they vote for president and vice president in a way to let us the people choose to keep our country safe and to help our community. So I think we should keep the electoral college.\n\nUnder t...",human,human,ai_vs_human_text,unknown,0.0,1583,283
73707,"Dear Senator, I know that you have many issues to think about and have a lot of decisions to make, but I think it is the subject of the Electoral College is a very important subject for you to ponder. The Electoral College needs to be changed, we need to vote for the presidency with the popular ...",human,human,ai_vs_human_text,unknown,0.0,3825,664
322557,"There is no definitive answer to this question as it depends on the individual. However, in general, young people are more likely to enjoy life than older people are.\n\nThere are several reasons for this. Eirstly, young people are typically more optimistic and energetic than older people. They ...",ai,unknown_ai,ai_vs_human_text,unknown,1.0,1054,174
66622,Many schools have established partnerships with companies to provide students with opportunities to explore various aspects of business. These partnerships are beneficial for students who have goals of starting their own businesses or who want to gain experience in the business world. By partner...,ai,unknown_ai,ai_vs_human_text,unknown,1.0,1479,220
109278,"My principle has decided that she wants everyone to participate in an extracurricular activity. In CY opinion, I believe it's the best for students to participate in school activity. It will be a great way to have fun, cake new friends, and get some exercise.\n\nMaking new friends is not always ...",human,human,ai_vs_human_text,unknown,0.0,3271,617
32988,The Electoral College is a system in the United States that is used to elect the President and Vice President of the country every four years. The question of whether it works or not is a subject of debate and can be analyzed in various ways.\n\nOne way to examine the efficacy of the Electoral C...,ai,unknown_ai,ai_vs_human_text,unknown,1.0,1940,320
62526,"In the world of education, there have been many arguments about which way of learning could benefit the most people. Some agree that the system we have now is the pinnacle of what we can do to help younger generations learn. However, there is definitely more we could do so the next firemen or po...",human,human,ai_vs_human_text,unknown,0.0,3279,586
313172,"The Face on Mars is just a natural landform. Why I say that because, aliens are not real, and they can't create the landforms that were seen. If there were aliens then it would have been more information found on NASA. If no human or thing can't breath in space an alien shouldn't be able to brea...",human,human,ai_vs_human_text,unknown,0.0,1045,197
436063,"Limit Of Car Usage\n\nWhen you think about limiting car usage you might think ""Well how am I supposed to get there. There's no way I'm able to get to my Destination without a car."" Yes, I'm pretty sure we all would'NT want to walk five thousand miles to visit a friend three hours away, but here ...",human,human,ai_vs_human_text,unknown,0.0,3003,591


## 3. Save processed snapshot ##

In [9]:
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(PROCESSED_DIR / "standardized_all.csv", index=False)

## 4. Binary SVM experiment ##

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

In [11]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

def evaluate_model(model, X_test, y_test, task_name):
    y_pred = model.predict(X_test)

    print(f"=== {task_name} ===")
    print(classification_report(y_test, y_pred))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))

    results = {
        "task": task_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "macro_f1": f1_score(y_test, y_pred, average="macro"),
        "weighted_f1": f1_score(y_test, y_pred, average="weighted"),
        "num_test": len(y_test),
    }

    return results, y_pred

In [12]:
binary_df = make_binary_dataset(df)

X = binary_df["text"]
y = binary_df["binary_label"]

print(binary_df.shape)
print(y.value_counts())

(478799, 6)
binary_label
human    285597
ai       193202
Name: count, dtype: Int64


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

In [14]:
binary_svm = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        max_features=100_000,
    )),
    ("clf", LinearSVC(
        C=1.0,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

In [15]:
binary_svm.fit(X_train, y_train)

y_pred = binary_svm.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

          ai       1.00      1.00      1.00     38640
       human       1.00      1.00      1.00     57120

    accuracy                           1.00     95760
   macro avg       1.00      1.00      1.00     95760
weighted avg       1.00      1.00      1.00     95760

[[38599    41]
 [   13 57107]]


In [16]:
binary_results = {
    "task": "binary_human_vs_ai",
    "model": "tfidf_linearsvc",
    "accuracy": accuracy_score(y_test, y_pred),
    "macro_f1": f1_score(y_test, y_pred, average="macro"),
    "weighted_f1": f1_score(y_test, y_pred, average="weighted"),
    "num_train": len(X_train),
    "num_test": len(X_test),
}

pd.DataFrame([binary_results]).to_csv(
    RESULTS_DIR / "binary_results.csv",
    index=False,
)

binary_results

{'task': 'binary_human_vs_ai',
 'model': 'tfidf_linearsvc',
 'accuracy': 0.9994360902255639,
 'macro_f1': 0.999414207861153,
 'weighted_f1': 0.9994360571206254,
 'num_train': 383039,
 'num_test': 95760}

In [17]:
error_df = pd.DataFrame({
    "text": X_test,
    "true_label": y_test,
    "predicted_label": y_pred,
})

errors = error_df[error_df["true_label"] != error_df["predicted_label"]]

print(errors.shape)
display(errors.sample(min(10, len(errors)), random_state=RANDOM_STATE))

(54, 3)


,text,true_label,predicted_label
473976,"Have you ever aspired to join NASA or become a scientist? Scientists have the opportunity to explore various parts of the world, including outer space. Have you ever had the desire to journey to Mars, also known as the ""Red Planet?"" Mars has an extensive history, and some NASA scientists claim t...",ai,human
254741,"NO MORE LIMITING!\n\nSome people believe that students should be required TT take a music, a drama, try an art class. But should students be forcibly required TT take a music, a drama, try an art class?\n\nStudents should NTT be required TT take a music try art classes, and make these classes as...",ai,human
351203,"Thomas Jefferson once said, ""Determine never ZO be idle...IZ is wonderful how much may be done if we are always doing."" Being active like pursuing career goals and helping others and being inactive can lead ZO a LOZ of health problems.\n\nDo people accomplish more if they are pursuing their own ...",ai,human
174233,"Have you ever made a awful mistake and think you can never come back from it? You can always come back from a mistake. Duke Ellington the famous jazz legend always said ""a problem is a chance to do your best because you learn from your mistakes.\n\nIf you mess up and make a mistake you will lear...",human,ai
473248,"No, I don't believe we should advocate for autonomous vehicles as they pose hazardous situations. \n\nIn such scenarios, the driver has no control whatsoever, rendering their presence in the vehicle pointless. This could potentially be very dangerous, as instead of the individual controlling the...",ai,human
166522,"First impressions are what I learn from many people. People have many impressions and yes, it is hard to change.\n\nImpressions are special to some people, but it also could affect people in certain ways. Ways where your whole behavior might change or attitude or maybe your whole person. When we...",ai,human
474024,"Have you ever fantasized about sitting back and unwinding while your vehicle does all the driving? Have you ever pondered the potential risks of such a scenario? I am of the opinion that semi or fully self-driving cars pose a threat to the world. These cars are not entirely autonomous, it's uncl...",ai,human
473285,"Dear Principal,\n\nI believe both of your proposed policies have their merits, however, to prevent students from getting into trouble, I feel that your first policy, which allows students to use their phones during lunch and free periods, is a better option. This policy also requires students to...",ai,human
473144,"Hello, I am known as Generic_Name and I am a student at Generic_School. Our principal, also named Generic_Name, has made a decision that every student must engage in at least one extracurricular activity. I strongly oppose this decision and believe that Generic_Name is mistaken. I believe that t...",ai,human
474739,"I argue in favor of keeping the Electoral College because it is a technicality that should be taken into consideration. We as a society are built around the idea ofvotes, and while it is possible to get by with only a popular vote, it is not possible to get by with only a popular vote. The Elect...",ai,human


In [18]:
binary_train, binary_test = train_test_split(
    binary_df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=binary_df["binary_label"],
)

X_train = binary_train["text"]
y_train = binary_train["binary_label"]

X_test = binary_test["text"]
y_test = binary_test["binary_label"]

In [19]:
binary_svm.fit(X_train, y_train)
y_pred = binary_svm.predict(X_test)

analysis_df = binary_test.copy()
analysis_df["predicted_label"] = y_pred
analysis_df["correct"] = analysis_df["binary_label"] == analysis_df["predicted_label"]

display(analysis_df.groupby("dataset_name")["correct"].mean())
display(analysis_df.groupby("source_model")["correct"].mean())
display(analysis_df.groupby("binary_label")["correct"].mean())

display(
    analysis_df[~analysis_df["correct"]][
        ["text", "binary_label", "predicted_label", "dataset_name", "source_model", "domain"]
    ].sample(10, random_state=RANDOM_STATE)
)

dataset_name
ai_text_detection_dataset    0.989071
ai_vs_human_text             0.999688
daigt_v2_train                    1.0
multi_model_detection        0.968085
Name: correct, dtype: Float64

source_model
Claude-4_Opus                              1.0
GPT-5_Preview                              1.0
Gemini-2_Pro                               1.0
NousResearch/Llama-2-7b-chat-hf            1.0
cohere-command                             1.0
darragh_claude_v6                          1.0
human                                 0.999772
llama2_chat                                1.0
mistralai/Mistral-7B-Instruct-v0.1         1.0
palm-text-bison1                           1.0
unknown_ai                             0.99892
Name: correct, dtype: Float64

binary_label
ai       0.998939
human    0.999772
Name: correct, dtype: Float64

,text,binary_label,predicted_label,dataset_name,source_model,domain
473976,"Have you ever aspired to join NASA or become a scientist? Scientists have the opportunity to explore various parts of the world, including outer space. Have you ever had the desire to journey to Mars, also known as the ""Red Planet?"" Mars has an extensive history, and some NASA scientists claim t...",ai,human,ai_text_detection_dataset,unknown_ai,unknown
254741,"NO MORE LIMITING!\n\nSome people believe that students should be required TT take a music, a drama, try an art class. But should students be forcibly required TT take a music, a drama, try an art class?\n\nStudents should NTT be required TT take a music try art classes, and make these classes as...",ai,human,ai_vs_human_text,unknown_ai,unknown
351203,"Thomas Jefferson once said, ""Determine never ZO be idle...IZ is wonderful how much may be done if we are always doing."" Being active like pursuing career goals and helping others and being inactive can lead ZO a LOZ of health problems.\n\nDo people accomplish more if they are pursuing their own ...",ai,human,ai_vs_human_text,unknown_ai,unknown
174233,"Have you ever made a awful mistake and think you can never come back from it? You can always come back from a mistake. Duke Ellington the famous jazz legend always said ""a problem is a chance to do your best because you learn from your mistakes.\n\nIf you mess up and make a mistake you will lear...",human,ai,ai_vs_human_text,human,unknown
473248,"No, I don't believe we should advocate for autonomous vehicles as they pose hazardous situations. \n\nIn such scenarios, the driver has no control whatsoever, rendering their presence in the vehicle pointless. This could potentially be very dangerous, as instead of the individual controlling the...",ai,human,ai_text_detection_dataset,unknown_ai,unknown
166522,"First impressions are what I learn from many people. People have many impressions and yes, it is hard to change.\n\nImpressions are special to some people, but it also could affect people in certain ways. Ways where your whole behavior might change or attitude or maybe your whole person. When we...",ai,human,ai_vs_human_text,unknown_ai,unknown
474024,"Have you ever fantasized about sitting back and unwinding while your vehicle does all the driving? Have you ever pondered the potential risks of such a scenario? I am of the opinion that semi or fully self-driving cars pose a threat to the world. These cars are not entirely autonomous, it's uncl...",ai,human,ai_text_detection_dataset,unknown_ai,unknown
473285,"Dear Principal,\n\nI believe both of your proposed policies have their merits, however, to prevent students from getting into trouble, I feel that your first policy, which allows students to use their phones during lunch and free periods, is a better option. This policy also requires students to...",ai,human,ai_text_detection_dataset,unknown_ai,unknown
473144,"Hello, I am known as Generic_Name and I am a student at Generic_School. Our principal, also named Generic_Name, has made a decision that every student must engage in at least one extracurricular activity. I strongly oppose this decision and believe that Generic_Name is mistaken. I believe that t...",ai,human,ai_text_detection_dataset,unknown_ai,unknown
474739,"I argue in favor of keeping the Electoral College because it is a technicality that should be taken into consideration. We as a society are built around the idea ofvotes, and while it is possible to get by with only a popular vote, it is not possible to get by with only a popular vote. The Elect...",ai,human,ai_text_detection_dataset,unknown_ai,unknown


Note on the above result: this is on the unbalanced, combined dataset. As you can see from the section 1 sanity check, most of the training/testing examples come from one dataset accounting for 97% of the combined dataset. 

### Per-dataset performance for random split

In [36]:
for dataset_name, subset in analysis_df.groupby("dataset_name"):
    print(f"\n=== {dataset_name} ===")
    print(f"Support: {len(subset)}")
    print(classification_report(
        subset["binary_label"],
        subset["predicted_label"]
    ))
    print("Confusion matrix:")
    print(confusion_matrix(
        subset["binary_label"],
        subset["predicted_label"]
    ))


=== ai_text_detection_dataset ===
Support: 2013
              precision    recall  f1-score   support

          ai       1.00      0.99      0.99      2013
       human       0.00      0.00      0.00         0

    accuracy                           0.99      2013
   macro avg       0.50      0.49      0.50      2013
weighted avg       1.00      0.99      0.99      2013

Confusion matrix:
[[1991   22]
 [   0    0]]

=== ai_vs_human_text ===
Support: 93059


c:\Users\Rinta\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Rinta\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Rinta\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

          ai       1.00      1.00      1.00     35960
       human       1.00      1.00      1.00     57099

    accuracy                           1.00     93059
   macro avg       1.00      1.00      1.00     93059
weighted avg       1.00      1.00      1.00     93059

Confusion matrix:
[[35941    19]
 [   10 57089]]

=== daigt_v2_train ===
Support: 594
              precision    recall  f1-score   support

          ai       1.00      1.00      1.00       594

    accuracy                           1.00       594
   macro avg       1.00      1.00      1.00       594
weighted avg       1.00      1.00      1.00       594

Confusion matrix:
[[594]]

=== multi_model_detection ===
Support: 94
              precision    recall  f1-score   support

          ai       0.96      1.00      0.98        73
       human       1.00      0.86      0.92        21

    accuracy                           0.97        94
   macro avg       0.98    

c:\Users\Rinta\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:409: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


In [37]:
per_dataset_results = []

for dataset_name, subset in analysis_df.groupby("dataset_name"):
    per_dataset_results.append({
        "task": "binary_random_split",
        "dataset_name": dataset_name,
        "accuracy": accuracy_score(subset["binary_label"], subset["predicted_label"]),
        "macro_f1": f1_score(subset["binary_label"], subset["predicted_label"], average="macro"),
        "weighted_f1": f1_score(subset["binary_label"], subset["predicted_label"], average="weighted"),
        "support": len(subset),
    })

per_dataset_results_df = pd.DataFrame(per_dataset_results)
display(per_dataset_results_df)

per_dataset_results_df.to_csv(
    RESULTS_DIR / "binary_random_split_per_dataset_results.csv",
    index=False,
)

,task,dataset_name,accuracy,macro_f1,weighted_f1,support
0,binary_random_split,ai_text_detection_dataset,0.989071,0.497253,0.994505,2013
1,binary_random_split,ai_vs_human_text,0.999688,0.999671,0.999688,93059
2,binary_random_split,daigt_v2_train,1.000000,1.000000,1.000000,594
3,binary_random_split,multi_model_detection,0.968085,0.951471,0.967179,94


## 6. Cross-dataset evaluation ##

In [24]:
label_counts = (
    binary_df
    .groupby(["dataset_name", "binary_label"])
    .size()
    .unstack(fill_value=0)
)

display(label_counts)

binary_label,ai,human
dataset_name,,
ai_text_detection_dataset,10129,0
ai_vs_human_text,179783,285416
daigt_v2_train,2970,1
multi_model_detection,320,180


In [25]:
MIN_PER_CLASS = 50

eligible_binary_datasets = label_counts[
    (label_counts.get("ai", 0) >= MIN_PER_CLASS) &
    (label_counts.get("human", 0) >= MIN_PER_CLASS)
].index.tolist()

eligible_binary_datasets

['ai_vs_human_text', 'multi_model_detection']

In [33]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

LABELS = ["ai", "human"]

def run_cross_dataset_binary(train_dataset, test_dataset, min_per_class=50):
    train_df = binary_df[binary_df["dataset_name"] == train_dataset].copy()
    test_df = binary_df[binary_df["dataset_name"] == test_dataset].copy()

    train_counts = train_df["binary_label"].value_counts()
    test_counts = test_df["binary_label"].value_counts()

    print("Train label counts:")
    print(train_counts)
    print()
    print("Test label counts:")
    print(test_counts)

    # Skip if training set does not have enough examples from both classes
    if any(train_counts.get(label, 0) < min_per_class for label in LABELS):
        print(f"Skipping: training dataset does not have at least {min_per_class} examples per class.")
        return None

    # Skip if test set does not have enough examples from both classes
    if any(test_counts.get(label, 0) < min_per_class for label in LABELS):
        print(f"Skipping: test dataset does not have at least {min_per_class} examples per class.")
        return None

    X_train = train_df["text"]
    y_train = train_df["binary_label"]

    X_test = test_df["text"]
    y_test = test_df["binary_label"]

    model = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True,
            max_features=100_000,
        )),
        ("clf", LinearSVC(
            C=1.0,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ])

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print(classification_report(
        y_test,
        y_pred,
        labels=LABELS,
        zero_division=0,
    ))

    print("Confusion matrix:")
    print(confusion_matrix(
        y_test,
        y_pred,
        labels=LABELS,
    ))

    return {
        "train_dataset": train_dataset,
        "test_dataset": test_dataset,
        "accuracy": accuracy_score(y_test, y_pred),
        "macro_f1": f1_score(y_test, y_pred, average="macro"),
        "weighted_f1": f1_score(y_test, y_pred, average="weighted"),
        "num_train": len(train_df),
        "num_test": len(test_df),
        "train_ai": train_counts.get("ai", 0),
        "train_human": train_counts.get("human", 0),
        "test_ai": test_counts.get("ai", 0),
        "test_human": test_counts.get("human", 0),
    }

In [34]:
cross_results = []

for train_dataset in eligible_binary_datasets:
    for test_dataset in eligible_binary_datasets:
        if train_dataset != test_dataset:
            print(f"\n=== Train: {train_dataset} | Test: {test_dataset} ===")
            result = run_cross_dataset_binary(train_dataset, test_dataset)

            if result is not None:
                cross_results.append(result)

cross_results_df = pd.DataFrame(cross_results)
display(cross_results_df)

cross_results_df.to_csv(
    RESULTS_DIR / "cross_dataset_binary_results.csv",
    index=False,
)


=== Train: ai_vs_human_text | Test: multi_model_detection ===
Train label counts:
binary_label
human    285416
ai       179783
Name: count, dtype: Int64

Test label counts:
binary_label
ai       320
human    180
Name: count, dtype: Int64
              precision    recall  f1-score   support

          ai       0.66      0.99      0.79       320
       human       0.79      0.08      0.15       180

    accuracy                           0.66       500
   macro avg       0.72      0.54      0.47       500
weighted avg       0.70      0.66      0.56       500

Confusion matrix:
[[316   4]
 [165  15]]

=== Train: multi_model_detection | Test: ai_vs_human_text ===
Train label counts:
binary_label
ai       320
human    180
Name: count, dtype: Int64

Test label counts:
binary_label
human    285416
ai       179783
Name: count, dtype: Int64
              precision    recall  f1-score   support

          ai       0.46      0.66      0.54    179783
       human       0.70      0.51      0.59  

,train_dataset,test_dataset,accuracy,macro_f1,weighted_f1,num_train,num_test,train_ai,train_human,test_ai,test_human
0,ai_vs_human_text,multi_model_detection,0.662000,0.469884,0.559240,465199,500,179783,285416,320,180
1,multi_model_detection,ai_vs_human_text,0.566618,0.565373,0.570655,500,465199,320,180,179783,285416


### AI-only transfer evaluation
Some datasets contain only AI-generated examples or are extremely imbalanced. These are not suitable for full binary cross-dataset evaluation, but they can still be used to measure AI recall.

In [28]:
def run_ai_only_transfer_test(train_dataset, test_dataset, min_train_per_class=50):
    train_df = binary_df[binary_df["dataset_name"] == train_dataset].copy()
    test_df = binary_df[
        (binary_df["dataset_name"] == test_dataset) &
        (binary_df["binary_label"] == "ai")
    ].copy()

    train_counts = train_df["binary_label"].value_counts()

    print("Train label counts:")
    print(train_counts)
    print()
    print("AI-only test size:")
    print(len(test_df))

    if any(train_counts.get(label, 0) < min_train_per_class for label in LABELS):
        print("Skipping: training dataset is not suitable for binary training.")
        return None

    if len(test_df) == 0:
        print("Skipping: no AI examples in test dataset.")
        return None

    X_train = train_df["text"]
    y_train = train_df["binary_label"]

    X_test = test_df["text"]
    y_test = test_df["binary_label"]

    model = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True,
            max_features=100_000,
        )),
        ("clf", LinearSVC(
            C=1.0,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ])

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    ai_recall = (y_pred == "ai").mean()

    print(f"AI recall on {test_dataset}: {ai_recall:.4f}")

    return {
        "train_dataset": train_dataset,
        "test_dataset": test_dataset,
        "task": "ai_only_transfer_recall",
        "ai_recall": ai_recall,
        "num_test_ai": len(test_df),
        "num_train": len(train_df),
    }

In [35]:
ai_only_results = []

for train_dataset in eligible_binary_datasets:
    for test_dataset in binary_df["dataset_name"].unique():
        if train_dataset != test_dataset:
            print(f"\n=== Train: {train_dataset} | AI-only Test: {test_dataset} ===")
            result = run_ai_only_transfer_test(train_dataset, test_dataset)

            if result is not None:
                ai_only_results.append(result)

ai_only_results_df = pd.DataFrame(ai_only_results)
display(ai_only_results_df)

ai_only_results_df.to_csv(
    RESULTS_DIR / "ai_only_transfer_results.csv",
    index=False,
)


=== Train: ai_vs_human_text | AI-only Test: daigt_v2_train ===
Train label counts:
binary_label
human    285416
ai       179783
Name: count, dtype: Int64

AI-only test size:
2970
AI recall on daigt_v2_train: 1.0000

=== Train: ai_vs_human_text | AI-only Test: ai_text_detection_dataset ===
Train label counts:
binary_label
human    285416
ai       179783
Name: count, dtype: Int64

AI-only test size:
10129
AI recall on ai_text_detection_dataset: 0.9232

=== Train: ai_vs_human_text | AI-only Test: multi_model_detection ===
Train label counts:
binary_label
human    285416
ai       179783
Name: count, dtype: Int64

AI-only test size:
320
AI recall on multi_model_detection: 0.9875

=== Train: multi_model_detection | AI-only Test: ai_vs_human_text ===
Train label counts:
binary_label
ai       320
human    180
Name: count, dtype: Int64

AI-only test size:
179783
AI recall on ai_vs_human_text: 0.6638

=== Train: multi_model_detection | AI-only Test: daigt_v2_train ===
Train label counts:
binary

,train_dataset,test_dataset,task,ai_recall,num_test_ai,num_train
0,ai_vs_human_text,daigt_v2_train,ai_only_transfer_recall,1.000000,2970,465199
1,ai_vs_human_text,ai_text_detection_dataset,ai_only_transfer_recall,0.923191,10129,465199
2,ai_vs_human_text,multi_model_detection,ai_only_transfer_recall,0.987500,320,465199
3,multi_model_detection,ai_vs_human_text,ai_only_transfer_recall,0.663834,179783,500
4,multi_model_detection,daigt_v2_train,ai_only_transfer_recall,0.428956,2970,500
5,multi_model_detection,ai_text_detection_dataset,ai_only_transfer_recall,0.852898,10129,500


## 7. Length-only baseline ##

In [29]:
binary_df = make_binary_dataset(df).copy()

binary_df["num_chars"] = binary_df["text"].str.len()
binary_df["num_words"] = binary_df["text"].str.split().str.len()

In [30]:
binary_train, binary_test = train_test_split(
    binary_df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=binary_df["binary_label"],
)

X_train_binary = binary_train["text"]
y_train_binary = binary_train["binary_label"]

X_test_binary = binary_test["text"]
y_test_binary = binary_test["binary_label"]

In [31]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

length_train = binary_train[["num_chars", "num_words"]]
length_test = binary_test[["num_chars", "num_words"]]

length_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        class_weight="balanced",
        random_state=RANDOM_STATE,
        max_iter=1000,
    )),
])

length_model.fit(length_train, y_train_binary)
length_pred = length_model.predict(length_test)

print(classification_report(y_test_binary, length_pred))
print(confusion_matrix(y_test_binary, length_pred))

              precision    recall  f1-score   support

          ai       0.70      0.77      0.74     38640
       human       0.84      0.78      0.81     57120

    accuracy                           0.78     95760
   macro avg       0.77      0.78      0.77     95760
weighted avg       0.78      0.78      0.78     95760

[[29872  8768]
 [12534 44586]]


In [32]:
length_results = {
    "task": "binary_length_only_baseline",
    "model": "logistic_regression_num_chars_num_words",
    "accuracy": accuracy_score(y_test, length_pred),
    "macro_f1": f1_score(y_test, length_pred, average="macro"),
    "weighted_f1": f1_score(y_test, length_pred, average="weighted"),
    "num_train": len(binary_train),
    "num_test": len(binary_test),
}

pd.DataFrame([length_results]).to_csv(
    RESULTS_DIR / "length_only_baseline_results.csv",
    index=False,
)

length_results

{'task': 'binary_length_only_baseline',
 'model': 'logistic_regression_num_chars_num_words',
 'accuracy': 0.7775480367585631,
 'macro_f1': 0.7721689805551631,
 'weighted_f1': 0.778924788536598,
 'num_train': 383039,
 'num_test': 95760}

## Multi-class AI model attribution
AI-only source model attribution: 

Given an AI-generated text, predict which model generated it. 

In [38]:
# Multi-class SVM experiment: AI model attribution
# Goal: among AI-generated examples with known source_model, predict the source model.

MIN_EXAMPLES_PER_CLASS = 75

multi_df = make_multiclass_dataset(
    df,
    min_examples_per_class=MIN_EXAMPLES_PER_CLASS
)

# Primary attribution task: only AI-generated examples
multi_ai_df = multi_df[multi_df["binary_label"] == "ai"].copy()

print("Multi-class AI-only dataset shape:", multi_ai_df.shape)
print("\nClass counts:")
display(multi_ai_df["source_model"].value_counts())

print("\nDataset counts:")
display(multi_ai_df["dataset_name"].value_counts())

print("\nSource model by dataset:")
display(pd.crosstab(multi_ai_df["source_model"], multi_ai_df["dataset_name"]))

Multi-class AI-only dataset shape: (3290, 6)

Class counts:


source_model
llama2_chat                           1754
darragh_claude_v6                      406
NousResearch/Llama-2-7b-chat-hf        297
palm-text-bison1                       196
cohere-command                         161
mistralai/Mistral-7B-Instruct-v0.1     156
Gemini-2_Pro                           118
GPT-5_Preview                          107
Claude-4_Opus                           95
Name: count, dtype: Int64


Dataset counts:


dataset_name
daigt_v2_train           2970
multi_model_detection     320
Name: count, dtype: Int64


Source model by dataset:


dataset_name,daigt_v2_train,multi_model_detection
source_model,,
Claude-4_Opus,0,95
GPT-5_Preview,0,107
Gemini-2_Pro,0,118
NousResearch/Llama-2-7b-chat-hf,297,0
cohere-command,161,0
darragh_claude_v6,406,0
llama2_chat,1754,0
mistralai/Mistral-7B-Instruct-v0.1,156,0
palm-text-bison1,196,0


In [39]:
multi_ai_df["num_chars"] = multi_ai_df["text"].str.len()
multi_ai_df["num_words"] = multi_ai_df["text"].str.split().str.len()

In [40]:
num_classes = multi_ai_df["source_model"].nunique()

if num_classes < 2:
    raise ValueError(
        f"Need at least 2 source_model classes for multi-class classification, "
        f"but found {num_classes}."
    )

In [41]:
multi_train, multi_test = train_test_split(
    multi_ai_df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=multi_ai_df["source_model"],
)

X_train_multi = multi_train["text"]
y_train_multi = multi_train["source_model"]

X_test_multi = multi_test["text"]
y_test_multi = multi_test["source_model"]

print("Train class counts:")
display(y_train_multi.value_counts())

print("Test class counts:")
display(y_test_multi.value_counts())

Train class counts:


source_model
llama2_chat                           1403
darragh_claude_v6                      325
NousResearch/Llama-2-7b-chat-hf        238
palm-text-bison1                       157
cohere-command                         129
mistralai/Mistral-7B-Instruct-v0.1     125
Gemini-2_Pro                            94
GPT-5_Preview                           85
Claude-4_Opus                           76
Name: count, dtype: Int64

Test class counts:


source_model
llama2_chat                           351
darragh_claude_v6                      81
NousResearch/Llama-2-7b-chat-hf        59
palm-text-bison1                       39
cohere-command                         32
mistralai/Mistral-7B-Instruct-v0.1     31
Gemini-2_Pro                           24
GPT-5_Preview                          22
Claude-4_Opus                          19
Name: count, dtype: Int64

In [42]:
multi_svm = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        max_features=100_000,
    )),
    ("clf", LinearSVC(
        C=1.0,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        max_iter=5000,
    )),
])

multi_svm.fit(X_train_multi, y_train_multi)
multi_pred = multi_svm.predict(X_test_multi)

In [43]:
multi_labels = sorted(y_test_multi.unique())

print(classification_report(
    y_test_multi,
    multi_pred,
    labels=multi_labels,
    zero_division=0,
))

multi_cm = confusion_matrix(
    y_test_multi,
    multi_pred,
    labels=multi_labels,
)

multi_cm_df = pd.DataFrame(
    multi_cm,
    index=[f"true_{label}" for label in multi_labels],
    columns=[f"pred_{label}" for label in multi_labels],
)

display(multi_cm_df)

                                    precision    recall  f1-score   support

                     Claude-4_Opus       1.00      1.00      1.00        19
                     GPT-5_Preview       1.00      1.00      1.00        22
                      Gemini-2_Pro       1.00      1.00      1.00        24
   NousResearch/Llama-2-7b-chat-hf       1.00      1.00      1.00        59
                    cohere-command       1.00      0.97      0.98        32
                 darragh_claude_v6       1.00      1.00      1.00        81
                       llama2_chat       1.00      1.00      1.00       351
mistralai/Mistral-7B-Instruct-v0.1       1.00      1.00      1.00        31
                  palm-text-bison1       0.97      1.00      0.99        39

                          accuracy                           1.00       658
                         macro avg       1.00      1.00      1.00       658
                      weighted avg       1.00      1.00      1.00       658



,pred_Claude-4_Opus,pred_GPT-5_Preview,pred_Gemini-2_Pro,pred_NousResearch/Llama-2-7b-chat-hf,pred_cohere-command,pred_darragh_claude_v6,pred_llama2_chat,pred_mistralai/Mistral-7B-Instruct-v0.1,pred_palm-text-bison1
true_Claude-4_Opus,19,0,0,0,0,0,0,0,0
true_GPT-5_Preview,0,22,0,0,0,0,0,0,0
true_Gemini-2_Pro,0,0,24,0,0,0,0,0,0
true_NousResearch/Llama-2-7b-chat-hf,0,0,0,59,0,0,0,0,0
true_cohere-command,0,0,0,0,31,0,0,0,1
true_darragh_claude_v6,0,0,0,0,0,81,0,0,0
true_llama2_chat,0,0,0,0,0,0,351,0,0
true_mistralai/Mistral-7B-Instruct-v0.1,0,0,0,0,0,0,0,31,0
true_palm-text-bison1,0,0,0,0,0,0,0,0,39


In [44]:
multi_results = {
    "task": "ai_model_attribution",
    "model": "tfidf_linearsvc",
    "min_examples_per_class": MIN_EXAMPLES_PER_CLASS,
    "accuracy": accuracy_score(y_test_multi, multi_pred),
    "macro_f1": f1_score(y_test_multi, multi_pred, average="macro"),
    "weighted_f1": f1_score(y_test_multi, multi_pred, average="weighted"),
    "num_train": len(multi_train),
    "num_test": len(multi_test),
    "num_classes": y_test_multi.nunique(),
}

multi_results_df = pd.DataFrame([multi_results])

display(multi_results_df)

multi_results_df.to_csv(
    RESULTS_DIR / "multiclass_ai_model_results.csv",
    index=False,
)

,task,model,min_examples_per_class,accuracy,macro_f1,weighted_f1,num_train,num_test,num_classes
0,ai_model_attribution,tfidf_linearsvc,75,0.99848,0.99683,0.998478,2632,658,9


In [45]:
multi_report_dict = classification_report(
    y_test_multi,
    multi_pred,
    labels=multi_labels,
    output_dict=True,
    zero_division=0,
)

multi_report_df = pd.DataFrame(multi_report_dict).transpose()

display(multi_report_df)

multi_report_df.to_csv(
    RESULTS_DIR / "multiclass_ai_model_classification_report.csv"
)

,precision,recall,f1-score,support
Claude-4_Opus,1.000000,1.000000,1.000000,19.00000
GPT-5_Preview,1.000000,1.000000,1.000000,22.00000
Gemini-2_Pro,1.000000,1.000000,1.000000,24.00000
NousResearch/Llama-2-7b-chat-hf,1.000000,1.000000,1.000000,59.00000
cohere-command,1.000000,0.968750,0.984127,32.00000
darragh_claude_v6,1.000000,1.000000,1.000000,81.00000
llama2_chat,1.000000,1.000000,1.000000,351.00000
mistralai/Mistral-7B-Instruct-v0.1,1.000000,1.000000,1.000000,31.00000
palm-text-bison1,0.975000,1.000000,0.987342,39.00000
accuracy,0.998480,0.998480,0.998480,0.99848


In [46]:
multi_cm_df.to_csv(
    RESULTS_DIR / "multiclass_ai_model_confusion_matrix.csv"
)

In [47]:
multi_analysis_df = multi_test.copy()
multi_analysis_df["predicted_source_model"] = multi_pred
multi_analysis_df["correct"] = (
    multi_analysis_df["source_model"] == multi_analysis_df["predicted_source_model"]
)

print("Per-class attribution accuracy:")
display(
    multi_analysis_df
    .groupby("source_model")["correct"]
    .agg(["mean", "count"])
    .sort_values("mean")
)

print("Per-dataset attribution accuracy:")
display(
    multi_analysis_df
    .groupby("dataset_name")["correct"]
    .agg(["mean", "count"])
    .sort_values("mean")
)

Per-class attribution accuracy:


,mean,count
source_model,,
cohere-command,0.96875,32
Claude-4_Opus,1.0,19
GPT-5_Preview,1.0,22
Gemini-2_Pro,1.0,24
NousResearch/Llama-2-7b-chat-hf,1.0,59
darragh_claude_v6,1.0,81
llama2_chat,1.0,351
mistralai/Mistral-7B-Instruct-v0.1,1.0,31
palm-text-bison1,1.0,39


Per-dataset attribution accuracy:


,mean,count
dataset_name,,
daigt_v2_train,0.998314,593
multi_model_detection,1.0,65


In [48]:
multi_errors = multi_analysis_df[~multi_analysis_df["correct"]].copy()

print("Number of multi-class errors:", len(multi_errors))

if len(multi_errors) > 0:
    confusion_pairs = (
        multi_errors
        .groupby(["source_model", "predicted_source_model"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )

    display(confusion_pairs)

    display(
        multi_errors[
            [
                "text",
                "source_model",
                "predicted_source_model",
                "dataset_name",
                "domain",
                "num_words",
            ]
        ].sample(min(10, len(multi_errors)), random_state=RANDOM_STATE)
    )
else:
    print("No multi-class errors found.")

Number of multi-class errors: 1


,source_model,predicted_source_model,count
0,cohere-command,palm-text-bison1,1


,text,source_model,predicted_source_model,dataset_name,domain,num_words
288264,"The author makes a compelling case for studying Venus despite the dangers it presents. Throughout the article, the author provides examples of the unique opportunities that studying Venus can provide and successfully argues that the potential benefits of such research outweigh the risks.\n\nThe ...",cohere-command,palm-text-bison1,daigt_v2_train,Exploring Venus,302


### Stricter version using only multi-model dataset

In [49]:
multi_model_only_df = multi_ai_df[
    multi_ai_df["dataset_name"] == "multi_model_detection"
].copy()

print(multi_model_only_df.shape)
display(multi_model_only_df["source_model"].value_counts())

(320, 8)


source_model
Gemini-2_Pro     118
GPT-5_Preview    107
Claude-4_Opus     95
Name: count, dtype: Int64

In [50]:
if multi_model_only_df["source_model"].nunique() >= 2:
    mm_train, mm_test = train_test_split(
        multi_model_only_df,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=multi_model_only_df["source_model"],
    )

    X_train_mm = mm_train["text"]
    y_train_mm = mm_train["source_model"]

    X_test_mm = mm_test["text"]
    y_test_mm = mm_test["source_model"]

    mm_svm = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True,
            max_features=100_000,
        )),
        ("clf", LinearSVC(
            C=1.0,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            max_iter=5000,
        )),
    ])

    mm_svm.fit(X_train_mm, y_train_mm)
    mm_pred = mm_svm.predict(X_test_mm)

    print(classification_report(
        y_test_mm,
        mm_pred,
        zero_division=0,
    ))

    mm_results = {
        "task": "ai_model_attribution_multimodel_dataset_only",
        "model": "tfidf_linearsvc",
        "accuracy": accuracy_score(y_test_mm, mm_pred),
        "macro_f1": f1_score(y_test_mm, mm_pred, average="macro"),
        "weighted_f1": f1_score(y_test_mm, mm_pred, average="weighted"),
        "num_train": len(mm_train),
        "num_test": len(mm_test),
        "num_classes": y_test_mm.nunique(),
    }

    display(pd.DataFrame([mm_results]))

    pd.DataFrame([mm_results]).to_csv(
        RESULTS_DIR / "multiclass_multimodel_dataset_only_results.csv",
        index=False,
    )
else:
    print("Skipping multi_model_detection-only attribution: fewer than 2 classes.")

               precision    recall  f1-score   support

Claude-4_Opus       1.00      1.00      1.00        19
GPT-5_Preview       1.00      1.00      1.00        21
 Gemini-2_Pro       1.00      1.00      1.00        24

     accuracy                           1.00        64
    macro avg       1.00      1.00      1.00        64
 weighted avg       1.00      1.00      1.00        64



,task,model,accuracy,macro_f1,weighted_f1,num_train,num_test,num_classes
0,ai_model_attribution_multimodel_dataset_only,tfidf_linearsvc,1.0,1.0,1.0,256,64,3


## Summary of results

In [51]:
summary_rows = []

summary_rows.append(binary_results)
summary_rows.append(length_results)
summary_rows.append(multi_results)

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

summary_df.to_csv(
    RESULTS_DIR / "svm_summary_results.csv",
    index=False,
)

,task,model,accuracy,macro_f1,weighted_f1,num_train,num_test,min_examples_per_class,num_classes
0,binary_human_vs_ai,tfidf_linearsvc,0.999436,0.999414,0.999436,383039,95760,NaN,NaN
1,binary_length_only_baseline,logistic_regression_num_chars_num_words,0.777548,0.772169,0.778925,383039,95760,NaN,NaN
2,ai_model_attribution,tfidf_linearsvc,0.998480,0.996830,0.998478,2632,658,75.0,9.0


In [52]:
display(cross_results_df.sort_values("macro_f1", ascending=False))

,train_dataset,test_dataset,accuracy,macro_f1,weighted_f1,num_train,num_test,train_ai,train_human,test_ai,test_human
1,multi_model_detection,ai_vs_human_text,0.566618,0.565373,0.570655,500,465199,320,180,179783,285416
0,ai_vs_human_text,multi_model_detection,0.662000,0.469884,0.559240,465199,500,179783,285416,320,180


## Notes for final report